# Cricbuzz LiveStats: Data Fetching & Database Pipeline

This practice notebook demonstrates:
1. Loading credentials securely from `.env`
2. Making REST API calls to Cricbuzz Cricket API via RapidAPI
3. Parsing live match data, scorecards, and top player stats
4. Storing fetched records into the local SQLite database
5. Verifying relational data with SQL queries in pandas

In [ ]:
import os
import sqlite3
import requests
import urllib3
import pandas as pd
from dotenv import load_dotenv

# Suppress SSL warnings if Windows local cert verification is bypassed
urllib3.disable_warnings(urllib3.exceptions.InsecureRequestWarning)

# 1. Load environment variables
load_dotenv("../.env")
api_key = os.getenv("CRICBUZZ_API_KEY", "")
api_host = os.getenv("CRICBUZZ_API_HOST", "cricbuzz-cricket.p.rapidapi.com")
base_url = os.getenv("CRICBUZZ_API_BASE_URL", "https://cricbuzz-cricket.p.rapidapi.com")

headers = {
    "x-rapidapi-key": api_key,
    "x-rapidapi-host": api_host
}

print(f"API Configured: {bool(api_key and api_host)}")
print(f"API Host: {api_host}")

## 2. Fetch Live & Recent Matches from Cricbuzz API

In [ ]:
url = f"{base_url.rstrip('/')}/matches/v1/live"
response = requests.get(url, headers=headers, timeout=10, verify=False)

if response.status_code == 200:
    data = response.json()
    print("Live Matches Payload Keys:", list(data.keys()))
    type_matches = data.get("typeMatches", [])
    print(f"Found {len(type_matches)} match categories.")
else:
    print(f"API Call Failed with HTTP {response.status_code}: {response.text[:200]}")

## 3. Fetch Top Player Statistics (e.g. Most Runs)

In [ ]:
stats_url = f"{base_url.rstrip('/')}/stats/v1/topstats/0"
stats_resp = requests.get(stats_url, headers=headers, params={"statsType": "mostRuns"}, timeout=10, verify=False)

if stats_resp.status_code == 200:
    stats_data = stats_resp.json()
    headers_list = stats_data.get("headers", [])
    raw_values = [v.get("values", []) for v in stats_data.get("values", [])]
    
    if raw_values and len(raw_values[0]) == len(headers_list) + 1:
        cols = ["Player ID"] + headers_list
        df_top_runs = pd.DataFrame(raw_values, columns=cols)
    else:
        df_top_runs = pd.DataFrame(raw_values, columns=headers_list)
        
    print("Top 5 Run Scorers from Live Cricbuzz API:")
    print(df_top_runs.head(5))
else:
    print(f"Stats request failed: HTTP {stats_resp.status_code}")

## 4. Connect to SQLite Database & Query Relational Stats

In [ ]:
db_path = "../data/cricbuzz_livestats.db"
conn = sqlite3.connect(db_path)

# Query top 5 players by runs in ODI from historical database
sql = """
SELECT 
    p.full_name AS player,
    p.country,
    SUM(b.runs_scored) AS total_runs,
    ROUND(AVG(b.runs_scored), 2) AS batting_average
FROM players p
JOIN batting_performances b ON b.player_id = p.player_id
JOIN innings i ON i.innings_id = b.innings_id
JOIN matches m ON m.match_id = i.match_id
WHERE m.format = 'ODI'
GROUP BY p.player_id
ORDER BY total_runs DESC
LIMIT 5;
"""

df_db_results = pd.read_sql_query(sql, conn)
conn.close()

print("Top 5 ODI Batters from Database:")
print(df_db_results)